In [ ]:
!pip install ifcopenshell
!pip install ifcopenshell openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 MB 16.6 MB/s eta 0:00:00


In [ ]:
from google.colab import files  # ✅ Import the files module

print("📁 Please upload your IFC file...")
uploaded = files.upload()  # Opens file picker in Colab

# Get the uploaded file name
ifc_path = list(uploaded.keys())[0]
print(f"✅ Loaded file: {ifc_path}")


📁 Please upload your IFC file...


Saving 2024-006.01-01-TP-05VN-M01-PUB.ifc to 2024-006.01-01-TP-05VN-M01-PUB.ifc
✅ Loaded file: 2024-006.01-01-TP-05VN-M01-PUB.ifc


In [ ]:
import ifcopenshell
# Load the IFC file
model = ifcopenshell.open(ifc_path)



In [ ]:
import ifcopenshell.geom
import pandas as pd
from google.colab import files

print(ifcopenshell.version)

0.8.5


In [ ]:
# -----------------------------
# USER SETTINGS
# -----------------------------

PROPERTY_SET_NAME = "Text"   # Property set to search for

# Parameters to extract from this property set
PARAMETERS_TO_EXTRACT = {
    "Elementname": "Pavadinimas",
    "NSIKcodeLF": "Funkcinės sistemos tipas",
    "NSIKcodeLT": "NcodePtID",
    "NSIKcodeLK": "NcodeLKtID"
}

In [ ]:
# -----------------------------
# FIND PROPERTY SETS
# -----------------------------
def get_property_sets(element):
    """Return all property sets attached to an element."""
    psets = {}
    for rel in element.IsDefinedBy:
        if rel.is_a("IfcRelDefinesByProperties"):
            prop = rel.RelatingPropertyDefinition
            if prop.is_a("IfcPropertySet"):
                psets[prop.Name] = prop
    return psets

# -----------------------------
# COLLECT PARAMETERS
# -----------------------------
rows = []

for element in model.by_type("IfcElement"):
    psets = get_property_sets(element)

    if PROPERTY_SET_NAME in psets:
        pset = psets[PROPERTY_SET_NAME]

        row = {"ElementType": element.is_a()}

        # Extract requested parameters
        for output_name, ifc_param_name in PARAMETERS_TO_EXTRACT.items():
            value = None
            for prop in pset.HasProperties:
                if prop.Name == ifc_param_name:
                    if hasattr(prop, "NominalValue") and prop.NominalValue:
                        value = prop.NominalValue.wrappedValue
                    else:
                        value = None

            # Store using YOUR output column name
            row[output_name] = value

        rows.append(row)

# -----------------------------
# MAKE TABLE
# -----------------------------
df_ifc = pd.DataFrame(rows)
df_ifc

,ElementType,Elementname,NSIKcodeLF,NSIKcodeLT,NSIKcodeLK
0,IfcFlowController,None,None,None,%HQB20
1,IfcFlowController,None,None,None,%HQB20
2,IfcFlowController,None,None,None,%BHA20
3,IfcFlowController,None,None,FA,%QMA10
4,IfcFlowController,None,None,FA,%QMA10
...,...,...,...,...,...
1330,IfcFlowTerminal,None,None,FD,%QMA10
1331,IfcFlowTerminal,None,None,FD,%QMA10
1332,IfcFlowTerminal,None,None,FD,%QMA10
1333,IfcFlowTerminal,None,None,FD,%QMA10


In [ ]:
import requests
import json

api_url = "https://nsik.planuojustatau.lt/api/getActiveVersion"

try:
    response = requests.get(api_url)
    response.raise_for_status() # Raise an exception for HTTP errors
    data = response.json()
except requests.exceptions.RequestException as e:
    print(f"Error querying API: {e}")

In [ ]:
import pandas as pd

def extract_all_levels(group_list):
    flat_data = []
    for group in group_list:
        current_level_data = {}
        if 'level1' in group: current_level_data['level1'] = group['level1']
        if 'name' in group: current_level_data['name_level1'] = group['name']

        if 'children' in group:
            for child1 in group['children']:
                child1_data = current_level_data.copy()
                if 'level2' in child1: child1_data['level2'] = child1['level2']
                if 'name' in child1: child1_data['name_level2'] = child1['name']

                if 'children' in child1:
                    for child2 in child1['children']:
                        child2_data = child1_data.copy()
                        if 'level3' in child2: child2_data['level3'] = child2['level3']
                        if 'name' in child2: child2_data['name_level3'] = child2['name']

                        if 'children' in child2:
                            for child3 in child2['children']:
                                child3_data = child2_data.copy()
                                if 'level4' in child3: child3_data['level4'] = child3['level4']
                                if 'name' in child3: child3_data['name_level4'] = child3['name']
                                flat_data.append(child3_data)
                        else:
                            flat_data.append(child2_data)
                else:
                    flat_data.append(child1_data)
        else:
            flat_data.append(current_level_data)
    return flat_data


all_flat_levels = []
if 'data' in data and isinstance(data['data'], list):
    for class_item in data['data']:
        if 'groups' in class_item and isinstance(class_item['groups'], list):
            all_flat_levels.extend(extract_all_levels(class_item['groups']))

print("API")
display(pd.DataFrame(all_flat_levels).fillna(''))

API


,level1,name_level1,level2,name_level2,level3,name_level3,level4,name_level4
0,A,Gyvenamųjų patalpų paskirčių tipo patalpos,AA,Gyvenamųjų patalpų paskirties grupės patalpa,AAA,Gyvenamųjų patalpų (butų) paskirties patalpa,%AAA10,Butas
1,A,Gyvenamųjų patalpų paskirčių tipo patalpos,AB,Įvairių socialinių grupių patalpų paskirties g...,ABA,Įvairių socialinių grupių patalpų paskirties p...,%ABA10,Bendrabutis
2,A,Gyvenamųjų patalpų paskirčių tipo patalpos,AB,Įvairių socialinių grupių patalpų paskirties g...,ABA,Įvairių socialinių grupių patalpų paskirties p...,%ABA20,Globos namai
3,A,Gyvenamųjų patalpų paskirčių tipo patalpos,AB,Įvairių socialinių grupių patalpų paskirties g...,ABA,Įvairių socialinių grupių patalpų paskirties p...,%ABA30,Šeimos namai
4,A,Gyvenamųjų patalpų paskirčių tipo patalpos,AB,Įvairių socialinių grupių patalpų paskirties g...,ABA,Įvairių socialinių grupių patalpų paskirties p...,%ABA40,Vienuolynas
...,...,...,...,...,...,...,...,...
1780,E,Statinio statybos rūšys,EA,Naujo statinio statyba,,,,
1781,E,Statinio statybos rūšys,EB,Statinio rekonstravimas,,,,
1782,E,Statinio statybos rūšys,EC,Statinio kapitalinis remontas,,,,
1783,E,Statinio statybos rūšys,ED,Statinio paprastasis remontas,,,,


In [ ]:
import pandas as pd
import re

API_ROWS = (554, 1737)

API_LEVEL_COLUMNS = ["level1", "level2", "level3", "level4"]
API_NAME_COLUMNS  = ["name_level1", "name_level2", "name_level3", "name_level4"]

IFC_MATCH_COLUMNS = ["NSIKcodeLF", "NSIKcodeLT", "NSIKcodeLK"]

# Normalization function (critical for matching)
def normalize(code):
    if not isinstance(code, str):
        code = str(code)

    code = code.strip()
    code = code.replace(" ", "")
    code = code.replace(".", "")
    code = code.replace("-", "")
    code = code.replace("_", "")
    code = re.sub(r'[%?]', '', code)

    return code


# ----------------------------------------------------
# 1. PREPARE API TABLE
# ----------------------------------------------------

df_api = pd.DataFrame(all_flat_levels).fillna('')
df_api = df_api.iloc[API_ROWS[0]:API_ROWS[1] + 1].reset_index(drop=True)

# Normalize API codes
for col in API_LEVEL_COLUMNS:
    if col in df_api.columns:
        df_api[col] = df_api[col].astype(str).apply(normalize)


# ----------------------------------------------------
# 2. CLEAN IFC TABLE
# ----------------------------------------------------

df_ifc_clean = df_ifc.copy()

for col in IFC_MATCH_COLUMNS:
    df_ifc_clean[col] = df_ifc_clean[col].astype(str).apply(normalize)


# ----------------------------------------------------
# 3. MATCHING FUNCTION
# ----------------------------------------------------

def match_api_name(code, api_df):
    if code == "":
        return ""

    for level, name_col in zip(API_LEVEL_COLUMNS, API_NAME_COLUMNS):
        if level in api_df.columns:
            match = api_df[api_df[level] == code]
            if not match.empty:
                return match.iloc[0].get(name_col, "")

    return ""


# ----------------------------------------------------
# 4. APPLY MATCHING TO IFC TABLE
# ----------------------------------------------------

for col in IFC_MATCH_COLUMNS:
    df_ifc_clean[col + "_name"] = df_ifc_clean[col].apply(
        lambda x: match_api_name(x, df_api)
    )
'''
df_ifc_clean

# ----------------------------------------------------
# 5. GROUP ROWS BY ALL MATCHING VALUES
# ----------------------------------------------------
'''
group_columns = []
for col in IFC_MATCH_COLUMNS:
    group_columns.append(col)
    group_columns.append(col + "_name")

df_grouped = (
    df_ifc_clean
    .groupby(group_columns, dropna=False)
    .size()
    .reset_index(name="count")
)

df_grouped


,NSIKcodeLF,NSIKcodeLF_name,NSIKcodeLT,NSIKcodeLT_name,NSIKcodeLK,NSIKcodeLK_name,count
0,None,,,,HQB20,Skysčių filtras,1
1,None,,FA,Apsaugos nuo viršįtampių komponentas,QMA10,Sklendė,2
2,None,,FD,,QMA10,Sklendė,11
3,None,,FD,,QNA20,Maišytuvas,2
4,None,,FD,,WMB20,Grindinio latakas,8
5,None,,FD,,XKA10,Plautuvė,1
6,None,,FD,,XKE10,Trapas su hidrauline užtvara,5
7,None,,FM,Priešgaisrinis komponentas,QMA10,Sklendė,2
8,None,,FMarbaFG,,QMA10,Sklendė,16
9,None,,GL,Nepertraukiamo perdavimo komponentas,WMB20,Grindinio latakas,5
